# pymefpp_run

This is a wrapper script to the mefpp engine to reduce the data that has been partioned and associated with the scripts

*pymefpp_partition_calibration
*pymefpp_partition_science
*pymefpp_associate

After running those scripts we end up with a  directory named reduced that contains the following tree structure:

* reduced/calibs
        * /bpm
        * /bias
        * /dome
        * /sky
    * /science
    
Below science we have all the science partition to be reduced.

This script will:

* take a science partition at a time
* execute the reduction cascade

The idea is to identify all biases, dome flats, sky flats that are needed for the full reduction of that science partitions, and only these, and reduce them. This permits a piecemeal approach to the work.


        
 

In [1]:
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
#import datetime
import os
#import shutil
#import subprocess
from datetime import datetime
#from pathlib import Path
from pprint import pprint

# here we import the local mefpp_run library
import sys
sys.path.insert(0,'/home/fselman/omegacam/mefpp/python')
from mefpp_run import *
sys.path
#mefpp_run.__file__

['/home/fselman/omegacam/mefpp/python',
 '/opt/pampelmuse_0.100.11.2',
 '/home/fselman/omegacam/mefpp/python',
 '/opt/anaconda3/lib/python36.zip',
 '/opt/anaconda3/lib/python3.6',
 '/opt/anaconda3/lib/python3.6/lib-dynload',
 '',
 '/opt/anaconda3/lib/python3.6/site-packages',
 '/opt/anaconda3/lib/python3.6/site-packages/IPython/extensions',
 '/home/fselman/.ipython']

In [2]:
# we set the data reduction directories -- LONG TIME TO RUN
raw_data_dir = '/home/lschmidt/fernando/fits/'
root_dir = '/home/fselman/local_storage/omegacam'
reduced_dir = root_dir+'/reduced/'
calib_dir = reduced_dir+'/calibs/'
reduced_science_dir = reduced_dir+'/science/'
mefpp_dir = '/home/fselman/omegacam/mefpp/' 

os.chdir(root_dir)
global cwd
cwd = os.getcwd()
print('cwd: ', cwd)

# depending on size of data set this line might be very slow
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))


cwd:  /homelocal2/fselman/omegacam
number of science partitions:  205


In [3]:
# associations for the science frame 
sci_partitions = []
for science_file in science_list:
    sci_partitions.append(ExposurePartition(science_file, 'SCIENCE'))


In [4]:
# select data to reduce
print('--------------------- science frame associations -------------------')
science_partition_list = sci_partitions
i = 0
for partition in science_partition_list:
    print(i, partition.base_path)
    i+=1

--------------------- science frame associations -------------------
0 /home/fselman/local_storage/omegacam/reduced//science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-11T01:24:08
1 /home/fselman/local_storage/omegacam/reduced//science/CN_Ori_055207p790_m052500p5/r_SDSS/2017-03-12T00:24:41
2 /home/fselman/local_storage/omegacam/reduced//science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-12T00:24:41
3 /home/fselman/local_storage/omegacam/reduced//science/CN_Ori_055207p790_m052500p5/NB_659/2017-03-11T01:24:08
4 /home/fselman/local_storage/omegacam/reduced//science/BX_Pup_075415p580_m241936p5/NB_659/2017-03-11T02:24:50
5 /home/fselman/local_storage/omegacam/reduced//science/BX_Pup_075415p580_m241936p5/NB_659/2017-03-13T02:47:17
6 /home/fselman/local_storage/omegacam/reduced//science/BX_Pup_075415p580_m241936p5/r_SDSS/2017-03-11T02:24:50
7 /home/fselman/local_storage/omegacam/reduced//science/BX_Pup_075415p580_m241936p5/r_SDSS/2017-03-13T02:47:17
8 /home/fselman/local_storage/omegacam/redu

In [5]:
# we do the association here
science_partition_list = sci_partitions[199:200]
for scipart in science_partition_list:

    bias_part_list = []
    dome_part_list = []
    sky_part_list = []
    sci_part_list = []

    # -------------- DATA ASSOCIATION SECTION ---------------
    print('scipart: '+scipart.base_path)
    sci_part_list.append(scipart)
    associateBPM(scipart, scipart)
    print('        bpm part:  '+scipart.bpmPartition.base_path)
    associateBias(scipart, scipart)
    print('        bias part:  '+scipart.biasPartition.base_path)
    bias_part_list.append(scipart.biasPartition)
    associateMFF(scipart, scipart)
    print('        MFF  part:  '+scipart.mffPartition.base_path)

    # associations for the MFF frame
    mffpart = scipart.mffPartition
    print('         mffpart: '+mffpart.base_path)
    associateBPM(mffpart, scipart)
    print('                  bpm  part:  '+mffpart.bpmPartition.base_path)
    associateBias(mffpart, scipart)
    print('                  bias part:  '+mffpart.biasPartition.base_path)
    bias_part_list.append(mffpart.biasPartition)
    associateDome(mffpart, scipart)
    print('                  dome part:  '+mffpart.domePartition.base_path)
    dome_part_list.append(mffpart.domePartition)
    associateSky(mffpart, scipart)
    print('                   sky part:  '+mffpart.skyPartition.base_path)
    sky_part_list.append(mffpart.skyPartition)

    # associations for the SKY frame
    skypart = mffpart.skyPartition
    print('  ---> skypart: '+skypart.base_path)
    associateBPM(skypart, scipart)
    print('                  bpm  part:  '+skypart.bpmPartition.base_path)
    associateBias(skypart, scipart)
    print('                  bias part:  '+skypart.biasPartition.base_path)
    bias_part_list.append(skypart.biasPartition)
    associateDome(skypart, scipart)
    print('                  dome part:  '+skypart.domePartition.base_path)
    dome_part_list.append(skypart.domePartition)

    # associations for the dome frame
    domepart = ExposurePartition(skypart.domePartition.frame, 'FLAT,DOME')
    print('  ---> domepart: '+domepart.base_path)
    associateBPM(domepart, scipart)
    print('                  bpm  part:  '+domepart.bpmPartition.base_path)
    associateBias(domepart, scipart)
    print('                  bias part:  '+domepart.biasPartition.base_path)
    bias_part_list.append(domepart.biasPartition)

    # --------------------- DATA REDUCTION SECTION ---------------------
    # here we reduce the bpm frames which for now is only a cp operation
    mefpp_reduce_bpm(mefpp_dir, calib_dir)

    # we start reducing the biases
    mefpp_reduce_bias(mefpp_dir, bias_part_list)

    # reduction of dome frames
    mefpp_reduce_dome(mefpp_dir, dome_part_list)

    # reduction of sky frames
    mefpp_reduce_sky(mefpp_dir, sky_part_list)

    # we reduce the science frames here
    mefpp_reduce_science(mefpp_dir, sci_part_list)
    
    # Normalize science. This will show a single face to the
    # procedures that follow-
    #
    #  - divide by EXPTIME
    #  - divide by avg gain: from MFF get gain vs average counts line get gain for counts = 1.0
    # add AIRMASS keyword
    # add PHOT_K keyword
    # add SATURATION keyword
    # add extinction K keyword
    # add PHOT_C keyword
    #
    # mefpp_normalize_science(mefpp, sci_part_list)



scipart: /home/fselman/local_storage/omegacam/reduced//science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17
        bpm part:  /homelocal2/fselman/omegacam/reduced/calibs/bpm
        bias part:  /homelocal2/fselman/omegacam/reduced/calibs/bias/574184/2017-07-05T10:35:47
        MFF  part:  /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26
         mffpart: /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26
                  bpm  part:  /homelocal2/fselman/omegacam/reduced/calibs/bpm
                  bias part:  /homelocal2/fselman/omegacam/reduced/calibs/bias/574184/2017-07-07T10:55:20
                  dome part:  /homelocal2/fselman/omegacam/reduced/calibs/dome/574184/r_SDSS/2017-07-06T10:20:27
                   sky part:  /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26
  ---> skypart: /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:2


success running nohup /home/fselman/omegacam/mefpp//bin/mefpp  --bias /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26/OMEGACAM_MB_1x1.fits  --dome /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26/OMEGACAM_DOMEFF_r_SDSS_1x1.fits  --sky /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26/sky.list --skysave   > /homelocal2/fselman/omegacam/reduced/calibs/sky/200253584/r_SDSS/2017-07-07T22:21:26/mefpp.log
/home/fselman/local_storage/omegacam/reduced//science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17/OMEGACAM_MFF_r_SDSS_1x1.fits exists
/home/fselman/local_storage/omegacam/reduced//science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17 True True True True True
/home/fselman/local_storage/omegacam/reduced//science/WW_Cet_001124p770_m112843p2/r_SDSS/2017-07-05T06:58:17 True True True True True

 nohup /home/fselman/omegacam/mefpp//bin/mefpp  --bias /home/fselman/local